# Qwen3-4B LoRA: 5 CL-запусков по `pythoncodes_cl_scored`

Цель ноутбука: нормально и воспроизводимо запустить пять curriculum-learning вариантов LoRA для `pythoncodes`:

1. `cl_length`
2. `cl_perplexity`
3. `cl_lexical`
4. `cl_semantic`
5. `cl_llm_judge`

Ноутбук не содержит «магии обучения» внутри ячеек. Он управляет уже вынесенным кодом из `src.train.lora_train`: проверяет окружение, проверяет датасет, выбирает ровно нужные CL-запуски, запускает обучение, собирает train/eval-логи в pandas-таблицы и сохраняет их в `outputs/`.


## 0. Главные переключатели

Перед запуском на кластере обычно достаточно поменять только эти значения.

`RUN_MODE = "full"` запускает все 5 CL-экспериментов. Для короткой проверки поставь `"smoke"` — тогда будет один маленький запуск на 10 шагов.

`SELECTED_MODEL = "4b-thinking"` оставлен как в старом ноутбуке. Если хочешь именно instruct-вариант из `src/config.py`, поставь `"4b-instruct"`.


In [ ]:
import os
from pathlib import Path

# Локальный запуск в Docker/WSL.
# Если у тебя реально используется thinking-модель, поменяй на "4b-thinking".
SELECTED_MODEL = "4b-instruct"
MAX_TOKENS = 4096

# "smoke" — короткая проверка одного запуска.
# "full" — все выбранные CL-запуски.
RUN_MODE = "smoke"

FORCE_RETRAIN = False
INCLUDE_BASELINE_IN_EVAL = False

# Локально удобнее False.
# True имеет смысл только если всё уже скачано и ты сознательно хочешь offline.
OFFLINE_MODE = False

# Smoke eval: 5 задач с MBPP и 5 с HumanEval.
# Full eval: None.
EVAL_MAX_TASKS = 5 if RUN_MODE == "smoke" else None

CL_RUN_NAMES = [
    "cl_length",
    "cl_perplexity",
    "cl_lexical",
    "cl_semantic",
    "cl_llm_judge",
]

# vLLM-настройки для локальной RTX 4070 Ti Super 16 GB.
VLLM_MAX_MODEL_LEN = 2048
GEN_MAX_TOKENS = 768
VLLM_GPU_MEMORY_UTILIZATION = 0.90
VLLM_MAX_NUM_SEQS = 4
VLLM_MAX_LORA_RANK = 64

os.environ["SELECTED_MODEL"] = SELECTED_MODEL
os.environ["MAX_TOKENS"] = str(MAX_TOKENS)

os.environ["VLLM_MAX_MODEL_LEN"] = str(VLLM_MAX_MODEL_LEN)
os.environ["GEN_MAX_TOKENS"] = str(GEN_MAX_TOKENS)
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = str(VLLM_GPU_MEMORY_UTILIZATION)
os.environ["VLLM_MAX_NUM_SEQS"] = str(VLLM_MAX_NUM_SEQS)
os.environ["VLLM_MAX_LORA_RANK"] = str(VLLM_MAX_LORA_RANK)

offline_keys = [
    "HF_HUB_OFFLINE",
    "TRANSFORMERS_OFFLINE",
    "HF_DATASETS_OFFLINE",
]

if OFFLINE_MODE:
    for key in offline_keys:
        os.environ[key] = "1"
else:
    for key in offline_keys:
        os.environ.pop(key, None)

print("SELECTED_MODEL:", SELECTED_MODEL)
print("MAX_TOKENS:", MAX_TOKENS)
print("RUN_MODE:", RUN_MODE)
print("OFFLINE_MODE:", OFFLINE_MODE)
print("EVAL_MAX_TASKS:", EVAL_MAX_TASKS)

print({
    key: os.environ.get(key)
    for key in offline_keys
})

print("vLLM env:", {
    "VLLM_MAX_MODEL_LEN": os.environ.get("VLLM_MAX_MODEL_LEN"),
    "GEN_MAX_TOKENS": os.environ.get("GEN_MAX_TOKENS"),
    "VLLM_GPU_MEMORY_UTILIZATION": os.environ.get("VLLM_GPU_MEMORY_UTILIZATION"),
    "VLLM_MAX_NUM_SEQS": os.environ.get("VLLM_MAX_NUM_SEQS"),
    "VLLM_MAX_LORA_RANK": os.environ.get("VLLM_MAX_LORA_RANK"),
})

{'OFFLINE_MODE': False, 'VLLM_MAX_MODEL_LEN': '2048', 'GEN_MAX_TOKENS': '768', 'HF_HUB_OFFLINE': None, 'TRANSFORMERS_OFFLINE': None, 'HF_DATASETS_OFFLINE': None}


## 1. Найти корень репозитория и импортировать проект

Старый ноутбук ломался, если запускать его из папки `notebooks/`, потому что `Path.cwd()` мог указывать не на корень проекта. Здесь корень ищется по наличию `src/` и `configs/`.


In [ ]:
import sys
import json
import time
import shutil
from copy import deepcopy
from pathlib import Path

import pandas as pd

%load_ext autoreload
%autoreload 2


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for path in [start, *start.parents]:
        if (path / "src").exists() and (path / "configs").exists():
            return path

    raise RuntimeError(
        "Не нашёл корень проекта. Запусти ноутбук из master-cluster-HSE "
        "или положи его внутрь репозитория."
    )


PROJECT_DIR = find_project_root()
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Важно: env-переменные из первой ячейки должны быть выставлены ДО импорта src.config.
import src.config as config

from src.train.lora_train import (
    load_train_config,
    load_pythoncodes_cl_df,
    train_lora_experiments,
    collect_training_table,
    prepare_lora_experiments,
)

from src.inference.lora_eval import (
    prepare_eval_datasets,
    evaluate_lora_experiments,
    cleanup_vllm,
)

print("PROJECT_DIR:", PROJECT_DIR)
print("SELECTED_MODEL:", config.SELECTED_MODEL)
print("MODEL_NAME:", config.MODEL_NAME)
print("MODEL_PATH:", config.MODEL_PATH)
print("MAX_TOKENS:", config.MAX_TOKENS)
print("DATASETS_DIR:", config.DATASETS_DIR)
print("OUTPUTS_DIR:", config.OUTPUTS_DIR)
print("MODELS_DIR:", config.MODELS_DIR)

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
/workspace/src/train/lora_train.py:19: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 05-01 03:25:57 [gpt_oss_triton_kernels_moe.py:34] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton_kernels.routing'
🦥 Unsloth Zoo will now patch everything to make training faster!
PROJECT_DIR: /workspace
SELECTED_MODEL: 4b-instruct
MODEL_NAME: Qwen/Qwen3-4B-Instruct-2507
MODEL_PATH: /workspace/models/qwen3-4b-instruct-2507
DATASETS_DIR: /workspace/datasets
OUTPUTS_DIR: /workspace/outputs
MODELS_DIR: /workspace/models
MAX_TOKENS: 4096
OFFLINE_MODE: False


## 2. Проверить конфиг обучения

Ожидается файл `configs/train/lora_pythoncodes_cl.yaml`. Из него берутся LoRA-параметры, batch size, gradient accumulation, max steps, eval/logging steps и список curriculum-запусков.

Ноутбук дальше принудительно выбирает только пять CL-запусков из `CL_RUN_NAMES`, без обычного `sft_pythoncodes`.


In [ ]:
CONFIG_PATH = PROJECT_DIR / "configs" / "train" / "lora_pythoncodes_cl.yaml"

train_cfg = load_train_config(str(CONFIG_PATH))
runs_by_name = {run["name"]: run for run in train_cfg["runs"]}

missing_runs = [name for name in CL_RUN_NAMES if name not in runs_by_name]
if missing_runs:
    raise ValueError(f"В конфиге нет запусков: {missing_runs}")

cl_runs = [runs_by_name[name] for name in CL_RUN_NAMES]

print("CONFIG_PATH:", CONFIG_PATH)

display(pd.DataFrame(cl_runs))

print("training:")
display(pd.Series(train_cfg["training"]))

print("lora:")
display(pd.Series(train_cfg["lora"]))

print("dataset:")
display(pd.Series(train_cfg["dataset"]))

CONFIG_PATH: /workspace/configs/train/lora_pythoncodes_cl.yaml


,name,category_col
0,cl_length,length_category
1,cl_perplexity,ppl_category
2,cl_lexical,lexical_cluster_category
3,cl_semantic,semantic_cluster_category
4,cl_llm_judge,llm_judge_category


training:


train_batch_size                    1
gradient_steps                      8
max_steps                         600
stage_max_steps                   200
warmup_steps                       25
lr                             0.0002
lr_scheduler_type              cosine
optim                      adamw_8bit
assistant_only_loss              True
packing                         False
logging_steps                      10
eval_steps                         50
save_steps                        100
save_total_limit                    2
report_to                 tensorboard
dataloader_num_workers              2
dataloader_pin_memory            True
max_grad_norm                     1.0
dtype: object

lora:


r                                                                            32
alpha                                                                        32
dropout                                                                       0
bias                                                                       none
use_gradient_checkpointing                                              unsloth
target_modules                [q_proj, k_proj, v_proj, o_proj, gate_proj, up...
dtype: object

## 3. Проверить модель на диске

На compute-node интернета обычно нет, поэтому эта ячейка не скачивает модель. Она только показывает, существует ли локальная папка модели.

Если модель ещё не загружена, запускай следующую download-ячейку на login-node с интернетом и `OFFLINE_MODE = False`.


In [ ]:
model_path = Path(config.MODEL_PATH)

print("MODEL_NAME:", config.MODEL_NAME)
print("MODEL_PATH:", model_path)
print("exists:", model_path.exists())

if model_path.exists():
    files = sorted(p.name for p in model_path.iterdir())[:30]
    print("first files:", files)
else:
    raise FileNotFoundError(
        f"Модель не найдена локально: {model_path}\n"
        "Скачай её через download-ячейку ниже или положи модель в этот путь."
    )

MODEL_NAME: Qwen/Qwen3-4B-Instruct-2507
MODEL_PATH: /workspace/models/qwen3-4b-instruct-2507
exists: True
first files: ['.cache', '.gitattributes', 'LICENSE', 'README.md', 'config.json', 'generation_config.json', 'merges.txt', 'model-00001-of-00003.safetensors', 'model-00002-of-00003.safetensors', 'model-00003-of-00003.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


### Опционально: скачать модель на login-node

Не запускай эту ячейку на compute-node без интернета. По умолчанию `DO_DOWNLOAD_MODEL = False`.


In [ ]:
DO_DOWNLOAD_MODEL = False

if DO_DOWNLOAD_MODEL:
    if OFFLINE_MODE:
        raise RuntimeError(
            "OFFLINE_MODE=True. Для скачивания поставь OFFLINE_MODE=False "
            "и перезапусти kernel с первой ячейки."
        )

    from huggingface_hub import snapshot_download

    Path(config.MODEL_PATH).mkdir(parents=True, exist_ok=True)

    snapshot_download(
        repo_id=config.MODEL_NAME,
        local_dir=config.MODEL_PATH,
        local_dir_use_symlinks=False,
        resume_download=True,
    )

    print("downloaded:", config.MODEL_PATH)
else:
    print("Скачивание модели пропущено.")

Скачивание пропущено.


## 4. Загрузить и проверить `pythoncodes_cl_scored`

Ожидаемый распакованный датасет:

```text
datasets/pythoncodes_cl_scored/
  data-00000-of-00001.arrow
  state.json
  dataset_info.json
```

Если папки нет, код попробует взять parquet из `outputs/curriculum_array/pythoncodes_cl_scored.parquet`, как прописано в YAML.


In [ ]:
df = load_pythoncodes_cl_df(train_cfg)

required_category_cols = [
    "length_category",
    "ppl_category",
    "lexical_cluster_category",
    "semantic_cluster_category",
    "llm_judge_category",
]

missing_cols = [col for col in required_category_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"В датасете нет нужных CL-колонок: {missing_cols}")

print("rows:", len(df))
print("columns:", len(df.columns))

display(df.head(3))

coverage_rows = []

for col in required_category_cols:
    counts = df[col].value_counts(dropna=False).to_dict()

    coverage_rows.append({
        "column": col,
        "non_null": int(df[col].notna().sum()),
        "null": int(df[col].isna().sum()),
        "unique_non_null": sorted(str(x) for x in df[col].dropna().unique()),
        "counts": counts,
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

rows: 49626
columns: 19


,output,instruction,input,text,length_score,lexical_cluster_score,semantic_cluster_score,ppl_score,ifd_score,cond_loss,uncond_loss,llm_judge_score,llm_judge_raw,length_category,ppl_category,ifd_category,lexical_cluster_category,semantic_cluster_category,llm_judge_category
0,```python\ntasks = []\nwhile True:\n task =...,Help me set up my daily to-do list!,Setting up your daily to-do list...,Help me set up my daily to-do list! Setting up...,97.0,0.984718,12.805576,15.310144,1.797941,2.728516,1.517578,2.0,"forced_choice:2; logits={1: 51.5625, 2: 57.625...",medium,hard,medium,hard,medium,easy
1,```python\nshopping_list = {}\nwhile True:\n ...,Create a shopping list based on my inputs!,Creating a shopping list...,Create a shopping list based on my inputs! Cre...,109.0,0.982656,11.646426,14.298523,2.395778,2.660156,1.110352,2.0,"forced_choice:2; logits={1: 42.40625, 2: 54.81...",medium,hard,hard,hard,easy,easy
2,"```python\ntotal_time = 0\nfor i in range(1, 8...",Calculate how much time I spend on my phone pe...,Calculating weekly phone usage...,Calculate how much time I spend on my phone pe...,104.0,0.984558,11.865649,6.489057,1.376707,1.870117,1.358398,2.0,"forced_choice:2; logits={1: 42.625, 2: 55.625,...",medium,medium,easy,hard,medium,easy


,column,non_null,null,unique_non_null,counts
0,length_category,49626,0,"[easy, hard, medium]","{'hard': 16630, 'easy': 16510, 'medium': 16486}"
1,ppl_category,49626,0,"[easy, hard, medium]","{'hard': 16845, 'easy': 16407, 'medium': 16374}"
2,lexical_cluster_category,49626,0,"[easy, hard, medium]","{'hard': 16872, 'easy': 16378, 'medium': 16376}"
3,semantic_cluster_category,49626,0,"[easy, hard, medium]","{'hard': 16873, 'easy': 16377, 'medium': 16376}"
4,llm_judge_category,49626,0,"[easy, hard, medium]","{'easy': 27593, 'medium': 17452, 'hard': 4581}"


## 5. Подготовить список запусков

`FORCE_RETRAIN = False` означает: если у запуска уже есть `outputs/train_runs/<run>/summary.json` и LoRA-адаптер в `models/<selected_model>-<run>`, он будет пропущен. Это удобно после падения Slurm-задачи: можно просто перезапустить ноутбук/скрипт и добрать недостающие эксперименты.


In [ ]:
def run_paths(run_name):
    run_dir = Path(config.OUTPUTS_DIR) / "train_runs" / run_name
    adapter_dir = Path(config.MODELS_DIR) / f"{config.SELECTED_MODEL}-{run_name}"
    return run_dir, adapter_dir


def run_is_completed(run_name):
    run_dir, adapter_dir = run_paths(run_name)
    return (run_dir / "summary.json").exists() and adapter_dir.exists()


if RUN_MODE == "smoke":
    active_cfg = deepcopy(train_cfg)

    active_cfg["dataset"]["limit"] = 2000
    active_cfg["dataset"]["val_size"] = 100

    active_cfg["training"]["max_steps"] = 10
    active_cfg["training"]["stage_max_steps"] = 10
    active_cfg["training"]["eval_steps"] = 10
    active_cfg["training"]["save_steps"] = 10
    active_cfg["training"]["logging_steps"] = 1
    active_cfg["training"]["report_to"] = "none"

    selected_runs = [runs_by_name["cl_length"]]
else:
    active_cfg = deepcopy(train_cfg)
    selected_runs = cl_runs


if not FORCE_RETRAIN:
    before = len(selected_runs)

    selected_runs = [
        run for run in selected_runs
        if not run_is_completed(run["name"])
    ]

    skipped = before - len(selected_runs)
    print("skipped completed runs:", skipped)
else:
    print("FORCE_RETRAIN=True: завершённые запуски не пропускаются.")


selected_runs_df = pd.DataFrame(selected_runs)

print("RUN_MODE:", RUN_MODE)
print("selected runs:", [run["name"] for run in selected_runs])

display(selected_runs_df)

status_rows = []

for run_name in CL_RUN_NAMES:
    run_dir, adapter_dir = run_paths(run_name)

    status_rows.append({
        "run": run_name,
        "run_dir": str(run_dir),
        "adapter_dir": str(adapter_dir),
        "summary_exists": (run_dir / "summary.json").exists(),
        "adapter_exists": adapter_dir.exists(),
        "completed": run_is_completed(run_name),
    })

status_df = pd.DataFrame(status_rows)
display(status_df)

RUN_MODE: smoke
FORCE_RETRAIN: False
skipped completed: 0
selected runs: ['cl_length']


,name,category_col,already_completed,will_run,run_dir,adapter_dir
0,cl_length,length_category,False,True,/workspace/outputs/train_runs/cl_length,/workspace/models/4b-instruct-cl_length


## 6. Запустить обучение

Эта ячейка запускает выбранные эксперименты. Для `RUN_MODE = "full"` это пять CL-вариантов; для `RUN_MODE = "smoke"` — короткая проверка `cl_length`.

Логи каждого запуска сохраняются в:

```text
outputs/train_runs/<run_name>/metrics.csv
outputs/train_runs/<run_name>/metrics.parquet
outputs/train_runs/<run_name>/summary.json
```

Адаптеры LoRA сохраняются в:

```text
models/<selected_model>-<run_name>/
```


In [ ]:
if len(selected_runs) == 0:
    print("Нет запусков для обучения: все выбранные эксперименты уже выглядят завершёнными.")
    train_result_df = pd.DataFrame()
else:
    started_at = time.time()

    train_result_df = train_lora_experiments(
        train_cfg=active_cfg,
        runs=selected_runs,
    )

    elapsed_min = (time.time() - started_at) / 60
    print(f"elapsed_min: {elapsed_min:.2f}")

    display(train_result_df)

train_result_df

=== Experiment 1/1: cl_length ===
03:26:02 | INFO | === cl_length ===
03:26:02 | INFO | model=/workspace/models/qwen3-4b-instruct-2507
03:26:02 | INFO | adapter=/workspace/models/4b-instruct-cl_length
03:26:02 | INFO | train=1900, val=100
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.57.1. vLLM: 0.16.1.dev0+g89a77b108.d20260417.cu128.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Building SFT prompts:   0%|          | 0/100 [00:00<?, ? examples/s]

03:26:24 | INFO | stage=easy; rows=1221


Building SFT prompts:   0%|          | 0/1221 [00:00<?, ? examples/s]

03:26:24 | WARNING | assistant_only_loss=True в YAML игнорируется: текущий train dataset имеет один текстовый столбец 'text', а не conversational messages. Для него безопаснее считать loss по полной строке.


ValueError: The specified `eos_token` ('<EOS_TOKEN>') is not found in the vocabulary of the given `processing_class` (Qwen2TokenizerFast). Ensure that the `eos_token` exists in the vocabulary before using it as an EOS token.

## 7. Собрать общую таблицу обучения

Здесь собираются `summary.json` и последние записи из `metrics.csv`. В таблице должны появляться `loss`, `eval_loss`, `grad_norm_manual`, `learning_rate`, `epoch` и служебные поля, если они были залогированы Trainer-ом.

Файлы сохраняются в:

```text
outputs/train_runs/summary.csv
outputs/train_runs/summary.parquet
```


In [ ]:
train_summary_df = collect_training_table()
display(train_summary_df)

summary_csv = Path(config.OUTPUTS_DIR) / "train_runs" / "summary.csv"
summary_parquet = Path(config.OUTPUTS_DIR) / "train_runs" / "summary.parquet"

print("saved:", summary_csv)
print("saved:", summary_parquet)

## 8. Собрать детальные train/eval/grad-логи в одну pandas-таблицу

Эта таблица нужна, чтобы потом нормально смотреть динамику loss, eval loss и gradient norm по всем экспериментам сразу.


In [ ]:
def read_metrics_for_runs(run_names):
    frames = []

    for run_name in run_names:
        run_dir, _ = run_paths(run_name)

        parquet_path = run_dir / "metrics.parquet"
        csv_path = run_dir / "metrics.csv"

        if parquet_path.exists():
            m = pd.read_parquet(parquet_path)
        elif csv_path.exists():
            m = pd.read_csv(csv_path)
        else:
            print(f"Нет metrics для {run_name}: {run_dir}")
            continue

        if "experiment" not in m.columns:
            m["experiment"] = run_name

        frames.append(m)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True, sort=False)


metrics_df = read_metrics_for_runs(CL_RUN_NAMES)

out_train_dir = Path(config.OUTPUTS_DIR) / "train_runs"
out_train_dir.mkdir(parents=True, exist_ok=True)

all_metrics_csv = out_train_dir / "all_cl_metrics.csv"
all_metrics_parquet = out_train_dir / "all_cl_metrics.parquet"

if len(metrics_df):
    metrics_df.to_csv(all_metrics_csv, index=False)
    metrics_df.to_parquet(all_metrics_parquet, index=False)

print("rows:", len(metrics_df))
print("saved:", all_metrics_csv)
print("saved:", all_metrics_parquet)

display(metrics_df.tail(20))

## 9. Быстрый просмотр ключевых метрик

Ячейка не обязательная, но удобная: она вытаскивает только шаги, loss, eval loss, grad norm и learning rate. Так проще понять, не взорвались ли градиенты и не развалился ли validation loss.


In [ ]:
metric_cols = [
    "experiment",
    "stage",
    "step",
    "epoch",
    "loss",
    "eval_loss",
    "grad_norm",
    "grad_norm_manual",
    "learning_rate",
]

existing_metric_cols = [
    col for col in metric_cols
    if col in metrics_df.columns
]

compact_metrics_df = (
    metrics_df[existing_metric_cols].copy()
    if len(metrics_df)
    else pd.DataFrame(columns=existing_metric_cols)
)

display(compact_metrics_df.tail(50))

compact_path = Path(config.OUTPUTS_DIR) / "train_runs" / "compact_cl_metrics.csv"
compact_metrics_df.to_csv(compact_path, index=False)

print("saved:", compact_path)

## 10. Подготовить список LoRA-адаптеров для inference/eval

Для evaluation берутся только реально существующие адаптеры. Если какой-то запуск ещё не дообучен, он будет показан в `missing_adapters_df`.


In [ ]:
eval_experiments = prepare_lora_experiments(
    include_base=INCLUDE_BASELINE_IN_EVAL,
    train_cfg=train_cfg,
)

experiments_df = pd.DataFrame(
    eval_experiments,
    columns=["experiment", "model_path", "adapter_path"],
)

existing_run_names = set(experiments_df["experiment"].tolist())

missing_adapters = []

for run_name in CL_RUN_NAMES:
    if run_name in existing_run_names:
        continue

    _, adapter_dir = run_paths(run_name)

    missing_adapters.append({
        "experiment": run_name,
        "expected_adapter_dir": str(adapter_dir),
        "exists": adapter_dir.exists(),
    })

missing_adapters_df = pd.DataFrame(missing_adapters)

display(experiments_df)
display(missing_adapters_df)

if len(eval_experiments) == 0:
    print("Нет адаптеров для evaluation.")

In [ ]:
DO_PREPARE_EVAL_DATASETS = True

if DO_PREPARE_EVAL_DATASETS:
    if OFFLINE_MODE:
        print(
            "OFFLINE_MODE=True. Если датасеты уже лежат локально, это нормально. "
            "Если нет — поставь OFFLINE_MODE=False, перезапусти kernel и запусти эту ячейку."
        )

    eval_data_status_df = prepare_eval_datasets(overwrite=False)
    display(eval_data_status_df)
else:
    print("Подготовка eval-датасетов пропущена.")

In [ ]:
import gc
import torch

for name in [
    "model",
    "trainer",
    "train_result",
    "train_ds",
    "val_ds",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

print("cuda_available:", torch.cuda.is_available())

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("free_gb:", free / 1024**3)
    print("total_gb:", total / 1024**3)

## 11. Запустить inference/eval и сохранить pandas-таблицу

`evaluate_lora_experiments` теперь лежит в `src/inference/lora_eval.py`, а не в `src/train/lora_train.py`.

Он не скачивает MBPP/HumanEval самовольно. По умолчанию он ждёт локальные датасеты:

```text
datasets/mbpp_sanitized_test
datasets/humaneval_test
```

Если их ещё нет, один раз запусти `prepare_eval_datasets()` с интернетом, потом можно снова включать `OFFLINE_MODE=True`.


In [ ]:
if len(eval_experiments) == 0:
    eval_df = pd.DataFrame()
else:
    out_name = "cl_eval_smoke" if EVAL_MAX_TASKS else "cl_eval_full"

    started_at = time.time()

    eval_df = evaluate_lora_experiments(
        eval_experiments,
        max_tasks=EVAL_MAX_TASKS,
        out_name=out_name,
        allow_download=False,
    )

    elapsed_min = (time.time() - started_at) / 60

    print(f"elapsed_min: {elapsed_min:.2f}")

    display(eval_df)

    eval_dir = Path(config.OUTPUTS_DIR) / "eval"

    print("saved:", eval_dir / f"{out_name}.csv")
    print("saved:", eval_dir / f"{out_name}.parquet")

eval_df

## 12. Финальная сводная таблица train + eval

Эта таблица удобна для отчёта: в ней каждая строка — эксперимент, рядом последние train-метрики и eval-метрики по benchmark-ам.


In [ ]:
if len(eval_experiments) == 0:
    eval_df = pd.DataFrame()
else:
    out_name = "cl_eval_smoke" if EVAL_MAX_TASKS else "cl_eval_full"

    started_at = time.time()

    eval_df = evaluate_lora_experiments(
        eval_experiments,
        max_tasks=EVAL_MAX_TASKS,
        out_name=out_name,
        allow_download=False,
    )

    elapsed_min = (time.time() - started_at) / 60

    print(f"elapsed_min: {elapsed_min:.2f}")

    display(eval_df)

    eval_dir = Path(config.OUTPUTS_DIR) / "eval"

    print("saved:", eval_dir / f"{out_name}.csv")
    print("saved:", eval_dir / f"{out_name}.parquet")

eval_df

In [ ]:
def make_final_summary(train_summary_df, eval_df):
    train_part = train_summary_df.copy()

    if len(eval_df):
        id_cols = [
            "experiment",
            "benchmark",
        ]

        skip_cols = [
            "model_path",
            "adapter_path",
        ]

        value_cols = [
            col for col in eval_df.columns
            if col not in id_cols + skip_cols
        ]

        eval_wide = eval_df.pivot_table(
            index="experiment",
            columns="benchmark",
            values=value_cols,
            aggfunc="first",
        )

        eval_wide.columns = [
            f"{metric}_{benchmark}"
            for metric, benchmark in eval_wide.columns
        ]

        eval_wide = eval_wide.reset_index()
    else:
        eval_wide = pd.DataFrame(columns=["experiment"])

    if "experiment" in train_part.columns:
        return train_part.merge(eval_wide, on="experiment", how="outer")

    return eval_wide


final_summary_df = make_final_summary(train_summary_df, eval_df)

final_dir = Path(config.OUTPUTS_DIR) / "eval"
final_dir.mkdir(parents=True, exist_ok=True)

final_csv = final_dir / "cl_train_eval_summary.csv"
final_parquet = final_dir / "cl_train_eval_summary.parquet"

final_summary_df.to_csv(final_csv, index=False)
final_summary_df.to_parquet(final_parquet, index=False)

display(final_summary_df)

print("saved:", final_csv)
print("saved:", final_parquet)

## 13. Мини-чеклист после запуска

После полного запуска должны существовать:

```text
models/4b-thinking-cl_length/
models/4b-thinking-cl_perplexity/
models/4b-thinking-cl_lexical/
models/4b-thinking-cl_semantic/
models/4b-thinking-cl_llm_judge/

outputs/train_runs/all_cl_metrics.csv
outputs/train_runs/all_cl_metrics.parquet
outputs/train_runs/summary.csv
outputs/train_runs/summary.parquet

outputs/eval/cl_eval_full.csv
outputs/eval/cl_eval_full.parquet
outputs/eval/cl_train_eval_summary.csv
outputs/eval/cl_train_eval_summary.parquet
```

Если используешь `SELECTED_MODEL = "4b-instruct"`, префикс папок будет `4b-instruct-*`.


In [ ]:
expected_paths = []

for run_name in CL_RUN_NAMES:
    _, adapter_dir = run_paths(run_name)

    expected_paths.append(adapter_dir)
    expected_paths.append(Path(config.OUTPUTS_DIR) / "train_runs" / run_name / "metrics.csv")
    expected_paths.append(Path(config.OUTPUTS_DIR) / "train_runs" / run_name / "summary.json")

expected_paths.extend([
    Path(config.OUTPUTS_DIR) / "train_runs" / "all_cl_metrics.csv",
    Path(config.OUTPUTS_DIR) / "train_runs" / "summary.csv",
    Path(config.DATASETS_DIR) / "mbpp_sanitized_test",
    Path(config.DATASETS_DIR) / "humaneval_test",
    Path(config.OUTPUTS_DIR) / "eval" / "cl_train_eval_summary.csv",
])

check_df = pd.DataFrame([
    {
        "path": str(path),
        "exists": path.exists(),
    }
    for path in expected_paths
])

display(check_df)